In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("chapter3").getOrCreate()

25/05/03 01:20:19 WARN Utils: Your hostname, talhas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.10.128.7 instead (on interface en0)
25/05/03 01:20:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/03 01:20:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
staticDataFrame = spark.read.format("csv")\
    .option("header","true")\
    .option("inferSchema","true")\
    .load("./Spark-The-Definitive-Guide/data/retail-data/by-day/*.csv")

In [3]:
staticDataFrame.createOrReplaceTempView("staticDataFrame")

In [4]:
staticSchema = staticDataFrame.schema

## 1. Group By Customer Id and Date to see which customers buys in which days most.

In [5]:
from pyspark.sql.functions import desc, window, column, col

heavyDays = staticDataFrame\
    .selectExpr(
        "CustomerID",
        "(Quantity * UnitPrice) as totalCost",
        "InvoiceDate"
    )\
    .groupBy(
        "CustomerID",
        window(col("InvoiceDate"),"1 day") 
    )\
    .sum("totalCost")\
    .orderBy(desc("sum(totalCost)"))

heavyDays.show(10)
    

[Stage 4:=============================>                            (5 + 5) / 10]

+----------+--------------------+------------------+
|CustomerID|              window|    sum(totalCost)|
+----------+--------------------+------------------+
|   17450.0|{2011-09-20 05:30...|          71601.44|
|      NULL|{2011-11-14 05:30...|          55316.08|
|      NULL|{2011-11-07 05:30...|          42939.17|
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2011-12-08 05:30...|31975.590000000007|
|   18102.0|{2011-09-15 05:30...|31661.540000000005|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2011-10-21 05:30...|          29693.82|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|   14646.0|{2011-10-20 05:30...|25833.559999999994|
+----------+--------------------+------------------+
only showing top 10 rows



## 2. Reducing number of partitions

In [6]:
spark.conf.set("spark.sql.shuffle.partitions",5)

## 3. Reading as Streaming input (1 file - 1 event)

In [7]:
streaminingDataFrame = spark.readStream\
    .schema(staticSchema)\
    .option("maxFilesPerTrigger",1)\
    .format("csv")\
    .option("header","true")\
    .load("./Spark-The-Definitive-Guide/data/retail-data/by-day/*.csv")


In [8]:
streaminingDataFrame.isStreaming
streaminingDataFrame.columns

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [9]:
purchaseByCustomerPerDay = streaminingDataFrame\
    .selectExpr(
        "CustomerID",
        "(Quantity * UnitPrice) as TotalCost",
        "InvoiceDate"
    )\
    .groupBy(
        col("CustomerID"),
        window(col("InvoiceDate"), "1 day")
    )\
    .sum("TotalCost")\
    .withColumnRenamed("sum(TotalCost)","TotalCost")\
    .orderBy(desc(col("TotalCost")))

purchaseByCustomerPerDay.explain(mode="formatted")

== Physical Plan ==
* Sort (12)
+- Exchange (11)
   +- * HashAggregate (10)
      +- StateStoreSave (9)
         +- * HashAggregate (8)
            +- StateStoreRestore (7)
               +- * HashAggregate (6)
                  +- Exchange (5)
                     +- * HashAggregate (4)
                        +- * Project (3)
                           +- * Filter (2)
                              +- StreamingRelation (1)


(1) StreamingRelation
Output [8]: [InvoiceNo#69, StockCode#70, Description#71, Quantity#72, InvoiceDate#73, UnitPrice#74, CustomerID#75, Country#76]
Arguments: FileSource[./Spark-The-Definitive-Guide/data/retail-data/by-day/*.csv], [InvoiceNo#69, StockCode#70, Description#71, Quantity#72, InvoiceDate#73, UnitPrice#74, CustomerID#75, Country#76]

(2) Filter [codegen id : 1]
Input [8]: [InvoiceNo#69, StockCode#70, Description#71, Quantity#72, InvoiceDate#73, UnitPrice#74, CustomerID#75, Country#76]
Condition : isnotnull(InvoiceDate#73)

(3) Project [codegen id : 1]


## 4. Writing Streaming Output (Memory)

In [10]:
writeMemory =  purchaseByCustomerPerDay.writeStream\
    .format("memory")\
    .queryName("customerPurchase1")\
    .outputMode("complete")\

    .start()

25/05/03 01:20:27 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/yb/hd4ftdt171lc608qd_gg86dc0000gn/T/temporary-7f68224b-90fe-4484-8789-0159b48d6d5b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/05/03 01:20:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [28]:
spark.sql(
    """
        SELECT * FROM customerPurchase1
    """
).show()

+----------+--------------------+------------------+
|CustomerID|              window|         TotalCost|
+----------+--------------------+------------------+
|   17450.0|{2011-09-20 05:30...|          71601.44|
|      NULL|{2011-11-14 05:30...|          55316.08|
|      NULL|{2011-11-07 05:30...|          42939.17|
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2011-12-08 05:30...|31975.590000000007|
|   18102.0|{2011-09-15 05:30...|31661.540000000005|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2011-10-21 05:30...|          29693.82|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|   14646.0|{2011-10-20 05:30...|25833.559999999994|
|      NULL|{2010-12-10 05:30...|25399.560000000012|
|      NULL|{2010-12-17 05:30...|25371.769999999768|
|      NULL|{2011-11-25 05:30...|24148.069999999992|
|      NULL|{2011-11-29 05:30...|23744.250000000055|
|   12415.0|{2011-06-15 05:30...| 23426.81000000001|
|      NULL|{2010-12-06 05:30...|23395.0999999

## 5. Wrirting Streaming data (Console) 

In [29]:
writeConsole =  purchaseByCustomerPerDay.writeStream\
    .format("Console")\
    .queryName("customerPurchase2")\
    .option("checkpointLocation", "./tmp/checkpoint_console")\
    .outputMode("complete")\
    .start()

25/05/03 01:36:12 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/05/03 01:36:13 WARN HDFSBackedStateStoreProvider: The state for version 162 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
25/05/03 01:36:13 WARN HDFSBackedStateStoreProvider: The state for version 162 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
25/05/03 01:36:13 WARN HDFSBackedStateStoreProvider: The state for version 162 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the first batch of starting query.
25/05/03 01:36:13 WARN HDFSBackedStateStoreProvider: The state for version 162 doesn't exist in loadedMaps. Reading snapshot file and delta files if needed...Note that this is normal for the

-------------------------------------------
Batch: 162
-------------------------------------------
+----------+--------------------+------------------+
|CustomerID|              window|         TotalCost|
+----------+--------------------+------------------+
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|      NULL|{2010-12-10 05:30...|25399.560000000012|
|      NULL|{2010-12-17 05:30...|25371.769999999768|
|   12415.0|{2011-06-15 05:30...| 23426.81000000001|
|      NULL|{2010-12-06 05:30...|23395.099999999904|
|      NULL|{2010-12-03 05:30...| 23021.99999999999|
|   15749.0|{2011-01-11 05:30...|           22998.4|
|   17450.0|{2011-01-11 05:30...|18620.199999999993|
|   14646.0|{2011-02-21 05:30...|18279.479999999996|
|   14646.0|{2011-03-29 05:30...|           18247.5|
|      NULL|{2011-05-10 05:30...| 17949.28000000001|
|   14156.0|{2011-01-14 05:30...|16774.719999999998|


-------------------------------------------
Batch: 163
-------------------------------------------
+----------+--------------------+------------------+
|CustomerID|              window|         TotalCost|
+----------+--------------------+------------------+
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|      NULL|{2010-12-10 05:30...|25399.560000000012|
|      NULL|{2010-12-17 05:30...|25371.769999999768|
|   12415.0|{2011-06-15 05:30...| 23426.81000000001|
|      NULL|{2010-12-06 05:30...|23395.099999999904|
|      NULL|{2010-12-03 05:30...| 23021.99999999999|
|   15749.0|{2011-01-11 05:30...|           22998.4|
|   17450.0|{2011-01-11 05:30...|18620.199999999993|
|   14646.0|{2011-02-21 05:30...|18279.479999999996|
|   14646.0|{2011-03-29 05:30...|           18247.5|
|      NULL|{2011-05-10 05:30...| 17949.28000000001|
|   14156.0|{2011-01-14 05:30...|16774.719999999998|


-------------------------------------------
Batch: 164
-------------------------------------------
+----------+--------------------+------------------+
|CustomerID|              window|         TotalCost|
+----------+--------------------+------------------+
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|      NULL|{2010-12-10 05:30...|25399.560000000012|
|      NULL|{2010-12-17 05:30...|25371.769999999768|
|   12415.0|{2011-06-15 05:30...| 23426.81000000001|
|      NULL|{2010-12-06 05:30...|23395.099999999904|
|      NULL|{2010-12-03 05:30...| 23021.99999999999|
|   15749.0|{2011-01-11 05:30...|           22998.4|
|   17450.0|{2011-01-11 05:30...|18620.199999999993|
|   14646.0|{2011-02-21 05:30...|18279.479999999996|
|   14646.0|{2011-03-29 05:30...|           18247.5|
|      NULL|{2011-05-10 05:30...| 17949.28000000001|
|   14156.0|{2011-01-14 05:30...|16774.719999999998|


-------------------------------------------
Batch: 165
-------------------------------------------
+----------+--------------------+------------------+
|CustomerID|              window|         TotalCost|
+----------+--------------------+------------------+
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|      NULL|{2010-12-10 05:30...|25399.560000000012|
|      NULL|{2010-12-17 05:30...|25371.769999999768|
|   12415.0|{2011-06-15 05:30...| 23426.81000000001|
|      NULL|{2010-12-06 05:30...|23395.099999999904|
|      NULL|{2010-12-03 05:30...| 23021.99999999999|
|   15749.0|{2011-01-11 05:30...|           22998.4|
|   17450.0|{2011-01-11 05:30...|18620.199999999993|
|   14646.0|{2011-02-21 05:30...|18279.479999999996|
|   14646.0|{2011-03-29 05:30...|           18247.5|
|      NULL|{2011-05-10 05:30...| 17949.28000000001|
|   14156.0|{2011-01-14 05:30...|16774.719999999998|


-------------------------------------------
Batch: 166
-------------------------------------------
+----------+--------------------+------------------+
|CustomerID|              window|         TotalCost|
+----------+--------------------+------------------+
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|      NULL|{2010-12-10 05:30...|25399.560000000012|
|      NULL|{2010-12-17 05:30...|25371.769999999768|
|   12415.0|{2011-06-15 05:30...| 23426.81000000001|
|      NULL|{2010-12-06 05:30...|23395.099999999904|
|      NULL|{2010-12-03 05:30...| 23021.99999999999|
|   15749.0|{2011-01-11 05:30...|           22998.4|
|   17949.0|{2011-06-30 05:30...|18854.780000000002|
|   17450.0|{2011-01-11 05:30...|18620.199999999993|
|   14646.0|{2011-02-21 05:30...|18279.479999999996|
|   14646.0|{2011-03-29 05:30...|           18247.5|
|      NULL|{2011-05-10 05:30...| 17949.28000000001|


-------------------------------------------
Batch: 167
-------------------------------------------
+----------+--------------------+------------------+
|CustomerID|              window|         TotalCost|
+----------+--------------------+------------------+
|      NULL|{2011-03-29 05:30...| 33521.39999999998|
|      NULL|{2010-12-21 05:30...|31347.479999999938|
|   18102.0|{2010-12-07 05:30...|          25920.37|
|      NULL|{2010-12-10 05:30...|25399.560000000012|
|      NULL|{2010-12-17 05:30...|25371.769999999768|
|   12415.0|{2011-06-15 05:30...| 23426.81000000001|
|      NULL|{2010-12-06 05:30...|23395.099999999904|
|      NULL|{2010-12-03 05:30...| 23021.99999999999|
|   15749.0|{2011-01-11 05:30...|           22998.4|
|   17949.0|{2011-06-30 05:30...|18854.780000000002|
|   17450.0|{2011-01-11 05:30...|18620.199999999993|
|   14646.0|{2011-02-21 05:30...|18279.479999999996|
|   14646.0|{2011-03-29 05:30...|           18247.5|
|      NULL|{2011-05-10 05:30...| 17949.28000000001|


In [31]:
spark.sql(
    """
        SELECT * FROM customerPurchase2
    """
).show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `customerPurchase2` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 2 pos 22;
'Project [*]
+- 'UnresolvedRelation [customerPurchase2], [], false


In [30]:
writeConsole.stop()

25/05/03 01:36:29 WARN TaskSetManager: Lost task 3.0 in stage 3520.0 (TID 354095) (10.10.128.7 executor driver): TaskKilled (Stage cancelled: Job 2096 cancelled part of cancelled job group 1d9da22a-4b93-4896-af3f-63f5c9f073a4)
25/05/03 01:36:29 WARN TaskSetManager: Lost task 4.0 in stage 3520.0 (TID 354096) (10.10.128.7 executor driver): TaskKilled (Stage cancelled: Job 2096 cancelled part of cancelled job group 1d9da22a-4b93-4896-af3f-63f5c9f073a4)
25/05/03 01:36:29 WARN TaskSetManager: Lost task 2.0 in stage 3520.0 (TID 354094) (10.10.128.7 executor driver): TaskKilled (Stage cancelled: Job 2096 cancelled part of cancelled job group 1d9da22a-4b93-4896-af3f-63f5c9f073a4)
25/05/03 01:36:29 WARN TaskSetManager: Lost task 0.0 in stage 3520.0 (TID 354092) (10.10.128.7 executor driver): TaskKilled (Stage cancelled: Job 2096 cancelled part of cancelled job group 1d9da22a-4b93-4896-af3f-63f5c9f073a4)
25/05/03 01:36:29 WARN TaskSetManager: Lost task 1.0 in stage 3520.0 (TID 354093) (10.10.128

## 6. Machine Learning and Advanced Analytics

In [ ]:
staticDataFrame.printSchema()